In [1]:
pip install python-dotenv

In [2]:
pip install snowflake-snowpark-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: cloudpickle
    Found existing installation: cloudpickle 3.1.1
    Uninstalling cloudpickle-3.1.1:
      Successfully uninstalled cloudpickle-3.1.1


In [3]:
pip install langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 10.4 MB/s eta 0:00:00


In [4]:
pip install langchain_deepseek

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.51
    Uninstalling langchain-core-0.3.51:
      Successfully uninstalled langchain-core-0.3.51


In [5]:
pip install gitingest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 698.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 8.0 MB/s eta 0:00:00


In [6]:
import os
from typing import Literal, List, Any
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.types import Command
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from langchain_core.prompts.chat import ChatPromptTemplate
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
import operator
import requests
from snowflake.snowpark import Session
from langchain_deepseek.chat_models import ChatDeepSeek
from gitingest import ingest
import re
#from utils.llms import LLMModel
#from toolkit.toolkits import *

In [7]:
from langchain_openai import ChatOpenAI

In [8]:
class AgentState(TypedDict):
    messages: Annotated[list[Any], add_messages]
    repo: str
    structure: str
    snowflake_repo: str
    #chat_history: list[BaseMessage]
    #intermediate_steps: Annotated[list[tuple[AgentAction, str]], operator.add]
    #count: int  # Tracks number of processed files
    files: List[str]
    filenames: List[str]
    processed_filenames: Annotated[List[str],  operator.add]
    summaries: Annotated[List[str],  operator.add]
    #current_filename: str
    #current_file: str
    next: str
    query: str
    current_reasoning: str


In [9]:




@tool("snowflake_repo_details")
def snowflake_repo_details(repo):
    """
    This function checks if the details of a repository are already present
    in the Snowflake database. The returned value is stored in snowflake_repo_value.
    """
    session = None
    try:
        # Create Snowflake session
        session = Session.builder.configs(connection_params).create()

        get_repo_name = f"""
        SELECT REPO_LINK FROM REPO_INFO WHERE REPO_LINK = '{repo}';
        """

        snowflake_repo = session.sql(get_repo_name).collect()

        if not snowflake_repo:
            print(f"No entry found for repo: {repo}")
            snowflake_repo_value = None
        else:
            snowflake_repo_value = snowflake_repo[0]["REPO_LINK"]
            print(snowflake_repo_value)

        return snowflake_repo_value

    except Exception as e:
        print(f"Error while fetching repo from Snowflake: {e}")
        return None

    finally:
        if session:
            try:
                session.close()
            except Exception as close_err:
                print(f"Error closing Snowflake session: {close_err}")




#@tool("snowflake_store_repo")
#def snowflake_store_repo(state: AgentState):
#    "This function is used to store the repository details in the snowflake database in the REPO_INFO table."
#
#    # Create Snowflake session
#    session = Session.builder.configs(connection_params).create()
#
#    repo = state["repo"]
#
#    repo_name = repo.split("/")[-1]
#    owner_name = repo.split("/")[-2]
#
#    repo_link = f"https://api.github.com/repos/{owner_name}/{repo_name}/commits"
#
#    # Fetch commits (latest first)
#    response = requests.get(repo_link, params={"per_page": 1})
#
#    if response.status_code == 200:
#        latest_commit_id = response.json()[0]["sha"]
#
#    #latest_commit_id = get_repo_latest_commit(state)
#    #created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#    #updated_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
#
#
#    insert_repo_table = f"""
#    INSERT INTO REPO_INFO (REPO_LINK, OWNER_NAME, REPO_NAME, LATEST_COMMIT_ID, CREATED_AT, UPDATED_AT)
#    VALUES ('{repo}', '{owner_name}', '{repo_name}', '{latest_commit_id}', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());
#    """
#
#    session.sql(insert_repo_table).collect()


#@tool("snowflake_store_repo")
#def snowflake_store_repo(state: AgentState):
#    """
#    This function is used to store the repository details in the Snowflake database
#    in the REPO_INFO table.
#    """
#    session = None
#    try:
#        # Create Snowflake session
#        session = Session.builder.configs(connection_params).create()
#
#        repo = state["repo"]
#        repo_name = repo.split("/")[-1]
#        owner_name = repo.split("/")[-2]
#        repo_link = f"https://api.github.com/repos/{owner_name}/{repo_name}/commits"
#
#        # Fetch commits (latest first)
#        response = requests.get(repo_link, params={"per_page": 1})
#
#        if response.status_code != 200:
#            raise Exception(f"Failed to fetch commits from GitHub: {response.status_code} - {response.text}")
#
#        response_data = response.json()
#
#        if not response_data:
#            raise Exception("No commit data returned from GitHub.")
#
#        latest_commit_id = response_data[0].get("sha")
#        if not latest_commit_id:
#            raise Exception("Latest commit ID not found in the GitHub response.")
#
#        insert_repo_table = f"""
#        INSERT INTO REPO_INFO (REPO_LINK, OWNER_NAME, REPO_NAME, LATEST_COMMIT_ID, CREATED_AT, UPDATED_AT)
#        VALUES ('{repo}', '{owner_name}', '{repo_name}', '{latest_commit_id}', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());
#        """
#
#        session.sql(insert_repo_table).collect()
#        return f"Repository '{repo}' successfully inserted into Snowflake."
#
#    except Exception as e:
#        return f"Error in snowflake_store_repo: {e}"
#
#    finally:
#        if session:
#            try:
#                session.close()
#            except Exception as close_err:
#                print(f"Error closing Snowflake session: {close_err}")
#




@tool("snowflake_store_file")
def snowflake_store_file(state: AgentState):
    "This function is used to store the file details in the snowflake database in the FILE_INFO table."

    #langgraph_logger.info('Started snowflake_store_file step')
    # Create Snowflake session
    session = Session.builder.configs(connection_params).create()

    repo = state["repo"]
    #langgraph_logger.info(f"Repo link in snowflake_store_file is '{repo}'")

    repo_name = repo.split("/")[-1]
    owner_name = repo.split("/")[-2]

    get_repo_id = f"""
    SELECT ID FROM REPO_INFO WHERE REPO_NAME = '{repo_name}' AND OWNER_NAME = '{owner_name}';
    """

    repo_id = session.sql(get_repo_id).collect()
    repo_id_value = repo_id[0]["ID"]

    #langgraph_logger.info(f"Repo ID fetched from REPO_INFO table in snowflake: '{repo_id_value}'")

    file_name = state['filenames']
    file_meaning = state['summaries']
    #file_meaning = state['current_file']
    escaped_text = f"$$\n{file_meaning}\n$$"

    insert_file_table = f"""
    INSERT INTO FILE_INFO (REPO_ID, FILE_NAME, FILE_MEANING, CREATED_AT, UPDATED_AT)
    VALUES ({repo_id_value}, '{file_name}', {escaped_text}, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());
    """

    session.sql(insert_file_table).collect()

    return f"File '{file_name}' successfully inserted into Snowflake."

    #langgraph_logger.info('Finished snowflake_store_file step')

In [10]:
@tool("file_content")
def file_content(state: AgentState):
    """
    This function is used to get the content of the current file that is in state.
    """

    file_name = state["current_filename"]

    file_index = state["filenames"].index(file_name)
    file_content = state["files"][file_index]

    return {"current_file": file_content}

In [11]:
@tool("summarizer_repo_file_details")
def summarizer_repo_file_details(state: AgentState):
    """
    This function gets the code files as text using the gitingest library. It filters out the specific files
    based on its extension and stores the required files in list (files). It also stores the file names as a list (filenames).
    """

    #langgraph_logger.info('Started repo_file_details step')

    repo = state["repo"]
    #langgraph_logger.info(f"Current repo name in repo_file_details is '{repo}'")

    summary, tree, content = ingest(repo)

    #repo_name = repo.split("/")[-1]
    #owner_name = repo.split("/")[-2]

    # Define the delimiter pattern (escape special characters)
    #file_pattern = r"\n=+\nFile: .+\n=+\n"
    file_pattern = r"(?=\n=+\nFile: .+\n=+\n)"

    # Split the content based on the delimiter
    files = re.split(file_pattern, content)

    # Define the regex pattern to match the filename
    filename_pattern = r"File: ([^\n]+)"

    # Find all filenames that match the pattern
    filenames = re.findall(filename_pattern, content)
    #print(filenames)

    for filename in filenames:
        if not filename.endswith(('.py', '.js', '.html', '.css', '.java', '.cpp', '.c', '.sh', '.bat', '.sql', '.db', '.sqlite', '.ipynb')):
            file_index = filenames.index(filename)
            files.pop(file_index)
            filenames.pop(file_index)

    #print(files)
    #print(filenames)
    #langgraph_logger.info('Finished repo_file_details step')
    #state.update({
    #    "files": files,
    #    "filenames": filenames,
    #    "processed_indexes": [],
    #    "structure": tree
    #})

    return {"structure":tree, "files": files, "filenames": filenames}

In [12]:
@tool("readme_builder")
def readme_builder(state: AgentState):
    """
    This function contains the structure required for the readme file.
    """
    readme_structure = """
    Below is the structure of README file.

    # Title

    This is an example file with maximal choices selected.

    This is a long description.

    ## Table of Contents

    - [Security](#security)
    - [Background](#background)
    - [Repository Structure](#Repository Structure)
    - [Install](#install)
    - [Usage](#usage)
    - [API](#api)
    - [Contributing](#contributing)
    - [License](#license)

    ## Security

    ### Any optional sections

    ## Background

    ### Any optional sections

    ## Repository Structure

    ## Install

    This module depends upon a knowledge of [Markdown]().

    ```
    ```

    ### Any optional sections

    ## Usage

    ```
    ```

    Note: The `license` badge image link at the top of this file should be updated with the correct `:user` and `:repo`.

    ### Any optional sections

    ## API

    ### Any optional sections

    ## More optional sections

    ## Contributing

    ### Any optional sections

    ## License
    """

    return readme_structure

In [13]:
members_dict = {'summarizer_node':'specialized agent to fetch the meaning of unstructured code files present in the project repository.',
                'finalizer_node':'specialized agent to generate the readme file for the project repository.',
                'snowflake_node':'specialized agent to store and fetch structured data from the snowflake database.'}

options = list(members_dict.keys()) + ["FINISH"]

worker_info = '\n\n'.join([f'WORKER: {member} \nDESCRIPTION: {description}' for member, description in members_dict.items()]) + '\n\nWORKER: FINISH \nDESCRIPTION: If proper readme file is generated and route to Finished'

system_prompt = (
    f"""You are a supervisor tasked with managing a conversation between the following workers. You cannot generate any content. All the required content must be generated by your assitants.
    ### SPECIALIZED ASSISTANT:
    {worker_info}
    Your primary role is to help the user create a comprehensive readme file for their entire project repository and judge the final readme file.
    If a customer requests to generate a readme file for their project, delegate the task to the appropriate specialized worker.
    Each worker will perform a task and respond with their results and status.

    You must store all the data to snowflake database only AFTER the readme file is generated.
    When all tasks are completed and the user query is resolved, respond with FINISH.

    **IMPORTANT RULES:**
    1. If the user's query is clearly answered and no further action is needed, respond with FINISH.
    2. If you detect repeated or circular conversations, or no useful progress after multiple turns, return FINISH.
    3. Always use previous context and results to determine if the user's intent has been satisfied. If it has — FINISH."""
)

In [14]:
#llm = ChatOpenAI(
#    model="gpt-4o",
#    openai_api_key='',
#    temperature=0
#)

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.7,
    max_tokens=8000,
    api_key=''
)

In [15]:
class Router(TypedDict):
    next: Literal["summarizer_node", "snowflake_node", "finalizer_node", "FINISH"]
    reasoning: str

In [16]:
def supervisor_node(state: AgentState) -> Command[Literal['summarizer_node', 'snowflake_node', 'finalizer_node', '__end__']]:
    print("**************************below is my state right after entering****************************")
    print(state)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"user's project repository is {state['repo']}. Generate a readme file."},
    ] + state["messages"]

    print("***********************this is my message*****************************************")
    print(messages)

    # query = state['messages'][-1].content if state["messages"] else ""
    query = ''
    if len(state['messages']) == 1:
        query = state['messages'][0].content

    print("************below is my query********************")
    print(query)

    response = llm.with_structured_output(Router).invoke(messages)

    goto = response["next"]

    print("********************************this is my goto*************************")
    print(goto)

    print("********************************")
    print(response["reasoning"])

    if goto == "FINISH":
        goto = END

    print("**************************below is my state****************************")
    print(state)

    if query:
        return Command(goto=goto, update={'next': goto,
                                        'query': query,
                                        'current_reasoning': response["reasoning"],
                                        'messages': [HumanMessage(content=f"user's project repository is {state['repo']}. Generate a readme file.")]
                        })
    return Command(goto=goto, update={'next': goto,
                                    'current_reasoning': response["reasoning"]}
                )

In [42]:
#def summarizer_node(state: AgentState) -> Command[Literal['supervisor']]:
#   print("***************** Called Summarizer Node ************")
#
#   ## Initialize file list if not present
#   #if "all_files" not in state:
#   #    file_data = summarizer_repo_file_details(state)
#   #    state.update({
#   #        "all_files": file_data["files"],
#   #        "filenames": file_data["filenames"],
#   #        "processed_indexes": []
#   #    })
#
#   # Find next unprocessed file
#   current_idx = next((
#       i for i in range(len(state["filenames"]))
#       if i not in state["processed_indexes"]
#   ), None)
#
#   if current_idx is None:
#       return Command(update=state, goto="supervisor")
#
#   # Get file content and name
#   file_content = state["files"][current_idx]
#   file_name = state["filenames"][current_idx]
#
#   # Generate summary
#   system_prompt = f"""
#   You are specialized agent to provide meaning of the code files present in
#   the project repository provided in the query. To get the files and filenames use the provided tool.
#   Analyze and summarize this code file: {file_name}. Give a comprehensive summary of the file that can be used
#   to understand the code. State all the important points. Do not miss any key details.
#   It should be detailed document which can be used to understand the entire code file.
#   After generating the meaning for each file, store the meaning in summaries list in state.
#   """
#
#
#   prompt = ChatPromptTemplate.from_messages([
#       ("system", system_prompt),
#       ("human", "File content:\n{file_content}"),
#       ("placeholder", "{messages}")
#   ])
#
#   summarizer_agent = create_react_agent(llm, tools=[summarizer_repo_file_details], system_prompt=prompt)
#   result = summarizer_agent.invoke({
#       "file_content": file_content,
#       "messages": state["messages"],
#       "structure": state["structure"]
#   })
#
#   # Update state
#   new_state = {
#       "processed_indexes": [*state["processed_indexes"], current_idx],
#       "summaries": [*state.get("summaries", []), {
#           "filename": file_name,
#           "summary": result["messages"][-1].content
#       }],
#       "messages": [*state["messages"],
#                   HumanMessage(content=f"Processed {file_name}"),
#                   AIMessage(content=result["messages"][-1].content)]
#   }
#
#   return Command(
#       update=new_state,
#       goto="supervisor"
#   )


In [24]:
def summarizer_node(state: AgentState) -> Command[Literal['supervisor']]:
    print("*****************called summarizer node************")

    system_prompt = """You are specialized agent to provide meaning of the code files present in
    the project repository provided in the query. You have to decide which files are important from the repository and summarize only those files.
    To get the file content and file names use the provided tool.
    Analyze and summarize each code file. Give a comprehensive summary of the file that can be used
    to understand the code. State all the important points. Do not miss any key details.
    It should be detailed document which can be used to understand the entire code file.
    After generating the meaning for each file, store the meaning in summaries list in state.
    You cannot generate the readme file. You have to generate just the summary of files.
    """

    system_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    system_prompt
                ),
                (
                    "placeholder",
                    "{messages}"
                ),
            ]
        )

    summarizer_agent = create_react_agent(model=llm,tools=[summarizer_repo_file_details, file_content], prompt=system_prompt)

    result = summarizer_agent.invoke(state)

    return Command(
        update={
            "messages": state["messages"] + [
                AIMessage(content=result["messages"][-1].content, name="summarizer_node")],
                #HumanMessage(content=result["messages"][-1].content, name="information_node"),
        },
        goto="supervisor",
    )

In [18]:
def snowflake_node(state: AgentState) -> Command[Literal['supervisor']]:
    print("*****************called snowflake node************")

    system_prompt = """
    You are a specialized agent to interact with the snowflake database. Fetch and store the required data
    from the database. You have access to multiple tools for this task.
    """

    system_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    system_prompt
                ),
                (
                    "placeholder",
                    "{messages}"
                ),
            ]
        )

    snowflake_agent = create_react_agent(model=llm,tools=[snowflake_repo_details, snowflake_store_file] ,prompt=system_prompt)

    result = snowflake_agent.invoke(state)

    return Command(
        update={
            "messages": state["messages"] + [
                AIMessage(content=result["messages"][-1].content, name="snowflake_node")
                #HumanMessage(content=result["messages"][-1].content, name="information_node")
            ]
        },
        goto="supervisor",
    )

In [19]:
def finalizer_node(state: AgentState) -> Command[Literal['supervisor']]:
    print("*****************called finalizer node************")

    system_prompt = """You are specialized agent to create a comprehensive readme file for the
    entire project repository based on the query. You have access to tool for this task. The tool contains a structure of the readme
    file for reference. You can include any or all
    of the points from the structure depending on the project files. Each section from the structure
    should have detailed information. The information shouldn't be just one liners. The information in the
    sections should not be generic or vague. Code snippets should be generated wherever they are absolutely
    necessary.

    """

    system_prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    system_prompt
                ),
                (
                    "placeholder",
                    "{messages}"
                ),
            ]
        )

    finalizer_agent = create_react_agent(model=llm, tools=[readme_builder], prompt=system_prompt)

    result = finalizer_agent.invoke(state)

    return Command(
        update={
            "messages": state["messages"] + [
                AIMessage(content=result["messages"][-1].content, name="finalizer_node")
                #HumanMessage(content=result["messages"][-1].content, name="booking_node")
            ]
        },
        goto="supervisor",
    )

In [20]:
graph = StateGraph(AgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("summarizer_node", summarizer_node)
graph.add_node("snowflake_node", snowflake_node)
graph.add_node("finalizer_node", finalizer_node)
graph.add_edge(START, "supervisor")
app = graph.compile()

In [59]:
#png_data = app.get_graph().draw_mermaid_png()

In [60]:
#from IPython.display import Image, display
#display(Image(app.get_graph().draw_mermaid_png()))

In [21]:
inputs = [
        HumanMessage(content='https://github.com/pratikkanade/SECFinancialStatementsSnowflake')
    ]

In [22]:
state = {
    "messages": inputs,
    "repo": str,
    "structure": str,
    "snowflake_repo": str,
    "files": [],
    "filenames": [],
    "processed_indexes": [],
    "summaries": [],
    #"current_filename": None,
    #"current_file": None
    "next": str,
    "query": str,
    "current_reasoning": str
}

In [25]:
result = app.invoke(state)

**************************below is my state right after entering****************************
{'messages': [HumanMessage(content='https://github.com/pratikkanade/SECFinancialStatementsSnowflake', additional_kwargs={}, response_metadata={}, id='285585b9-5611-4de8-a07a-4450532b5acd')], 'repo': <class 'str'>, 'structure': <class 'str'>, 'snowflake_repo': <class 'str'>, 'files': [], 'filenames': [], 'processed_filenames': [], 'summaries': [], 'next': <class 'str'>, 'query': <class 'str'>, 'current_reasoning': <class 'str'>}
***********************this is my message*****************************************
[{'role': 'system', 'content': "You are a supervisor tasked with managing a conversation between the following workers. You cannot generate any content. All the required content must be generated by your assitants.\n    ### SPECIALIZED ASSISTANT:\n    WORKER: summarizer_node \nDESCRIPTION: specialized agent to fetch the meaning of unstructured code files present in the project repository.\

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 65536 tokens. However, you requested 234848 tokens (226848 in the messages, 8000 in the completion). Please reduce the length of the messages or completion.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}